# Part 2 in code — embedding, memory, and categorical traces

Part 2 is where the paper stops waving its hands and starts giving us things we can actually compute.

The whole point is not to pretend we have the full phenomenal state.
The point is to define an operational approximation that does not immediately collapse into nonsense.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

results_dir = project_root / "results"
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

In [2]:
from gids_observer_framework.experiments.run_toy_equations import run_all_equation_checks

equation_checks = run_all_equation_checks()
part2_checks = equation_checks[equation_checks["paper_part"].str.contains("Part 2", regex=False)]
part2_checks[["paper_section", "test_name", "status", "metric", "interpretation"]]

,paper_section,test_name,status,metric,interpretation
0,"Dimensionality Reduction, Attention, and Relevance",salience_slice,PASS,"z=[0.30000000000000004, -2.0, 0.0, 1.7999999999999998]",Elementwise weighting behaves exactly as written; zero salience zeros the coordinate.
1,The Notion of State / Towards a Universal State Transition Function / Operational Definition of State,predictive_sufficiency,PASS,"max ΔP(Y|H,x) within same q,x = 0.000000",Different histories that compress to the same q give the same future law under the same proposition in this toy proc...
2,Memory as a Series of Vectors,memory_field_update,PASS,"m_t=[0.5, 0.8] -> m_t+1=[0.589, 0.778]",Memory updates smoothly when the trace weights are changed by context and similarity.
3,Categorical Trace Pooling as an Operational Memory Estimator,contextual_lift,PASS,"naive_conflict=1, typed_conflict=0",Typing the asymmetry removes a false contradiction in the dummy example.
4,Categorical Trace Pooling as an Operational Memory Estimator,event_categorical_pooling,PASS,"event_dim=12, active_slots=3",Average pooling plus null vectors and masks keeps the event representation fixed-width.
5,Categorical Trace Pooling as an Operational Memory Estimator,slow_regime_bank,PASS,"founder_head=[0.9, 0.1, 1.0], manager_mask=0.0",Regime-aware averaging and null fallback behave the way the paper says they should.
6,Categorical Trace Pooling as an Operational Memory Estimator,fast_task_conditioned_pool,PASS,"alpha_sum=1.000, max_alpha=0.484","The weights normalize to one and prioritize the more recent, more decisive trace."
7,Deriving the Transcendental Embedding,slow_embedding_estimator,PASS,"T_hat=[0.018, 0.553]",The first-pass estimator is a clean weighted sum rather than a sign-confused mixture of additions and subtractions.


## Salience slice

This is the algebraic version of a simple claim:
the next transition usually depends on a weighted slice of the person-space, not the whole thing at once.

In [3]:
from gids_observer_framework.embedding import salience_slice

projected = np.array([1.5, -2.0, 0.5, 3.0])
attention = np.array([0.2, 1.0, 0.0, 0.6])
pd.DataFrame({
    "projected_coordinate": projected,
    "attention_weight": attention,
    "active_coordinate": salience_slice(projected, attention),
})

,projected_coordinate,attention_weight,active_coordinate
0,1.5,0.2,0.3
1,-2.0,1.0,-2.0
2,0.5,0.0,0.0
3,3.0,0.6,1.8


## Contextual lifting

The paper is insistent about one thing here, and correctly so:
not every surface opposition is a deep contradiction.

First type the asymmetry. Then decide whether the conflict survives.

In [4]:
from gids_observer_framework.categorical import contextual_lift

raw_tokens = [
    {"family": "risk", "label": "low", "axis": "self"},
    {"family": "risk", "label": "high", "axis": "other"},
]
typed_tokens = contextual_lift(raw_tokens, context={"regime": "founder"})
pd.DataFrame(typed_tokens)

,family,label,axis,typed_token
0,risk,low,self,risk::self::low
1,risk,high,other,risk::other::high


## Within-event categorical pooling

This is the recommender move repurposed for observer-state estimation:
sparse categorical IDs become dense vectors and then get pooled into a fixed-width event representation.

In [5]:
from gids_observer_framework.toy_data import default_embedding_tables, FAMILIES, SOURCES
from gids_observer_framework.categorical import build_event_categorical_embedding

embedding_tables, null_vectors, projections = default_embedding_tables()
token_bags = {
    ("topic", "bio"): ["ai", "security"],
    ("topic", "behavior"): [],
    ("objection", "bio"): ["price"],
    ("objection", "behavior"): ["timing", "price"],
}

event_vec, slot_masks = build_event_categorical_embedding(
    token_bags=token_bags,
    embedding_tables=embedding_tables,
    null_vectors=null_vectors,
    projections=projections,
    family_order=FAMILIES,
    source_order=SOURCES,
)

pd.DataFrame({
    "slot": [f"{fam}/{src}" for fam in FAMILIES for src in SOURCES],
    "mask": list(slot_masks.values()),
}), event_vec[:12]

(                 slot  mask
 0           topic/bio   1.0
 1      topic/behavior   0.0
 2       objection/bio   1.0
 3  objection/behavior   1.0,
 array([ 0.69317731,  0.53520384,  1.        ,  0.        ,  0.        ,
         0.        ,  1.00527071, -0.3465234 ,  1.        ,  0.15642606,
         0.21188428,  1.        ]))

## Slow regime-aware bank and fast task-conditioned pool

The slow bank answers:
*what is this person generally like now across regimes?*

The fast pool answers:
*what is active now for this task given recent weighted history?*

In [6]:
from gids_observer_framework.categorical import build_slow_bank, build_fast_pool
from gids_observer_framework.toy_data import EVENT_CAT_DIM, REGIMES

history = [
    {"t": 0, "rho": "founder", "e_cat": np.array([1.0, 0.0, 1.0] + [0.0] * (EVENT_CAT_DIM - 3)), "beta_slow": 0.8, "task_relevance": 0.9, "action_intensity": 1.0, "weak_exposure": 0.2, "susceptibility": 1.2, "source_reliability": 0.9},
    {"t": 1, "rho": "founder", "e_cat": np.array([0.5, 0.5, 1.0] + [0.0] * (EVENT_CAT_DIM - 3)), "beta_slow": 0.2, "task_relevance": 0.8, "action_intensity": 0.1, "weak_exposure": 2.0, "susceptibility": 1.2, "source_reliability": 0.8},
    {"t": 2, "rho": "buyer",   "e_cat": np.array([0.0, 1.0, 1.0] + [0.0] * (EVENT_CAT_DIM - 3)), "beta_slow": 1.0, "task_relevance": 0.4, "action_intensity": 0.0, "weak_exposure": 3.0, "susceptibility": 0.5, "source_reliability": 0.6},
]

slow_bank = build_slow_bank(history, regimes=REGIMES, event_dim=EVENT_CAT_DIM)
fast_pool, alpha = build_fast_pool(history, event_dim=EVENT_CAT_DIM, current_time=3)

pd.DataFrame({
    "object": ["slow_bank_head", "fast_pool_head", "alpha_sum", "max_alpha"],
    "value": [
        slow_bank[:10].round(3).tolist(),
        fast_pool[:10].round(3).tolist(),
        float(alpha.sum()),
        float(alpha.max()),
    ],
})

,object,value
0,slow_bank_head,"[0.9, 0.1, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
1,fast_pool_head,"[0.67, 0.33, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,alpha_sum,1.0
3,max_alpha,0.484239


## Operational reading

The important part is not that these exact toy vectors are sacred.

The important part is that the project now has explicit hooks for:
- slow durable person structure
- fast recent task-conditioned structure
- source-aware categorical trace handling
- regime-aware memory
- falsifiable ablations later

That is the bridge from Part 2’s prose to reusable engineering.